## Observação metodológica

Este mapa mostra distribuição territorial do perfil do eleitorado, não o resultado da eleição por candidato. Quando houver a base oficial de votação por município, ela pode substituir essa visualização por um mapa de votos por candidato e por percentual do total de votos.

In [2]:
from pathlib import Path
import pandas as pd
import plotly.express as px

root = Path('..').resolve()

if 'geo' not in globals():
    profile = pd.read_parquet(root / 'data' / 'processed' / 'goias_profile_2026.parquet')
    import geobr
    import unicodedata

    def normalize_text(value):
        text = str(value).strip().upper()
        text = unicodedata.normalize('NFKD', text)
        text = ''.join(ch for ch in text if not unicodedata.combining(ch))
        return text

    profile['municipio_norm'] = profile['nm_municipio'].map(normalize_text)
    shape = geobr.read_municipality(code_muni='GO', year=2020)
    shape['municipio_norm'] = shape['name_muni'].map(normalize_text)
    geo = shape.merge(profile[['municipio_norm', 'total_eleitores', 'FEMININO', 'MASCULINO']], on='municipio_norm', how='left')
    geo['share_feminino'] = (geo['FEMININO'] / geo['total_eleitores'] * 100).fillna(0)

fig = px.choropleth(
    geo,
    geojson=geo.geometry.__geo_interface__,
    locations='code_muni',
    featureidkey='properties.code_muni',
    color='share_feminino',
    labels={'share_feminino': '% de eleitores do sexo feminino'},
    title='Goiás: proporção de eleitores do sexo feminino por município',
    projection='mercator'
)
fig.update_geos(fitbounds='locations', visible=False)
fig.update_layout(margin={'r': 0, 't': 30, 'l': 0, 'b': 0})
fig.show()

output = root / 'data' / 'processed' / 'goias_map_profile.html'
fig.write_html(output)
print(output)

D:\_DataScience_\ELEIÇÕES 2022 e 2026\data\processed\goias_map_profile.html


In [3]:
import unicodedata


def normalize_text(value):
    text = str(value).strip().upper()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    return text

profile['municipio_norm'] = profile['nm_municipio'].map(normalize_text)
shape = geobr.read_municipality(code_muni='GO', year=2020)
shape['municipio_norm'] = shape['name_muni'].map(normalize_text)
geo = shape.merge(profile[['municipio_norm', 'total_eleitores', 'FEMININO', 'MASCULINO']], on='municipio_norm', how='left')
geo['share_feminino'] = (geo['FEMININO'] / geo['total_eleitores'] * 100).fillna(0)
print(geo.head())
print('Municipios no shapefile:', len(geo))

   code_muni           name_muni  code_state abbrev_state name_state  \
0  5200050.0     Abadia de Goiás        52.0           GO      Goiás   
1  5200100.0           Abadiânia        52.0           GO      Goiás   
2  5200134.0             Acreúna        52.0           GO      Goiás   
3  5200159.0           Adelândia        52.0           GO      Goiás   
4  5200175.0  Água Fria de Goiás        52.0           GO      Goiás   

   code_region   name_region    year  \
0          5.0  Centro-Oeste  2020.0   
1          5.0  Centro-Oeste  2020.0   
2          5.0  Centro-Oeste  2020.0   
3          5.0  Centro-Oeste  2020.0   
4          5.0  Centro-Oeste  2020.0   

                                            geometry      municipio_norm  \
0  POLYGON ((-49.44692 -16.71646, -49.4361 -16.72...     ABADIA DE GOIAS   
1  POLYGON ((-48.90741 -15.98947, -48.79639 -16.0...           ABADIANIA   
2  POLYGON ((-50.35234 -17.30262, -50.34955 -17.3...             ACREUNA   
3  POLYGON ((-50.19448

In [4]:
from pathlib import Path

import geobr
import pandas as pd
import plotly.express as px

root = Path('..').resolve()
profile = pd.read_parquet(root / 'data' / 'processed' / 'goias_profile_2026.parquet')
print(profile.head())
print('Linhas:', len(profile))

      nm_municipio  total_eleitores  total_secoes  total_zonas  FEMININO  \
0  ABADIA DE GOIÁS            15641            53            1    8107.0   
1        ABADIÂNIA            12614            39            1    6447.0   
2          ACREÚNA            15990            48            1    8158.0   
3        ADELÂNDIA             2971             8            1    1435.0   
4         ALEXÂNIA            21998            76            1   11349.0   

   MASCULINO  NÃO INFORMADO  100 anos ou mais  16 anos  17 anos  ...  \
0     7534.0            0.0               1.0     51.0    102.0  ...   
1     6167.0            0.0               0.0     50.0    112.0  ...   
2     7832.0            0.0               0.0     88.0    129.0  ...   
3     1536.0            0.0               1.0     15.0     20.0  ...   
4    10649.0            0.0               6.0     89.0    169.0  ...   

   95 a 99 anos  Inválida  ANALFABETO  ENSINO FUNDAMENTAL COMPLETO  \
0           8.0       0.0         324   

# Mapa geoespacial de Goiás

Este notebook monta a visualização geoespacial do perfil do eleitorado em Goiás e prepara a base para a comparação entre municípios e, futuramente, votos por candidato.